<a href="https://colab.research.google.com/github/johnwbstanger/CampGlowStick/blob/main/examples/My_First_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train a mini-LLM on your own emails

This notebook trains a small language model **from scratch**, and samples a continuation from your own prompt.

The model uses token embeddings, **causal multi-head self-attention with rotary position encoding (RoPE)**, a GELU feed-forward network, residual connections, layer normalization, and a vocabulary softmax. Each block maps an input of shape `(batch, T, d)` to an output of the same shape. Training and inference happen within your Python runtime (your computer for local Jupyter, or a Google-hosted runtime for Colab).

> **Expectations:** Training a new language model on a few KB will **not** produce ChatGPT-level writing. The goal is to make the architecture and next-token training procedure inspectable. A GPU helps, but the default model is small enough to experiment with on a laptop CPU. You can click on `Runtime`->`Change runtime type` and select a GPU runtime.



In [133]:
import random
import pandas as pd
from google.colab import files

import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import ByteLevelBPETokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


VOCAB_TARGET = 2048           # BPE vocabulary trained locally on your corpus
CONTEXT_LENGTH = 128         # tokens (BPE tokens, not words)
EMBED_DIM = 128              # d
NUM_HEADS = 4               # H; each head has 32 dimensions
NUM_LAYERS = 3              # number of transformer blocks
DROPOUT = 0.1
BATCH_SIZE = 16
TRAIN_STEPS = 250
EVAL_INTERVAL = 50
LEARNING_RATE = 3e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [134]:
# This opens a browser prompt to select and upload files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
emails = pd.read_csv(filename,header=None)
documents=emails.iloc[:,0]

if len(documents) < 8:
    raise ValueError("Too little text. Include more emails.")

Saving corpus.csv to corpus (9).csv


## 1. Split by email and train a local byte-level BPE tokenizer

We split **whole emails** into training and validation sets to reduce direct leakage across the split. The tokenizer is trained **only on the training emails**; its vocabulary is local to this notebook, rather than a pretrained GPT/ChatGPT vocabulary. Byte-level BPE can encode arbitrary input text without an unknown-word token.

We append an end-of-document token to each email. Concatenating emails is a simple language-modeling convenience, though it creates synthetic transitions between documents.

In [135]:
shuffled = documents.copy()
random.Random(SEED).shuffle(shuffled)
n_val = max(1, round(len(shuffled) * 0.1))
val_docs = shuffled[:n_val]
train_docs = shuffled[n_val:]

special_tokens = ["<|endoftext|>"]
tokenizer = ByteLevelBPETokenizer()
tokenizer.train_from_iterator(
    train_docs,
    vocab_size=VOCAB_TARGET,
    min_frequency=2,
    special_tokens=special_tokens,
)
J = tokenizer.get_vocab_size()       # vocabulary size in our established notation
EOS_ID = tokenizer.token_to_id("<|endoftext|>")

def concatenate_encoded_docs(docs):
    ids = []
    for doc in docs:
        ids.extend(tokenizer.encode(doc).ids)
        ids.append(EOS_ID)
    return torch.tensor(ids, dtype=torch.long)

train_ids = concatenate_encoded_docs(train_docs)
val_ids = concatenate_encoded_docs(val_docs)
block_size = min(CONTEXT_LENGTH, len(train_ids) - 2, len(val_ids) - 2)
if block_size < 16:
    raise ValueError("Not enough tokenized text for training/validation. Fetch more messages.")
print(f"Training emails: {len(train_docs)} | Validation emails: {len(val_docs)}")
print(f"Vocabulary J: {J:,} | Train tokens: {len(train_ids):,} | Validation tokens: {len(val_ids):,}")
print(f"Context T: {block_size} | Token embedding d: {EMBED_DIM}")

Training emails: 1192 | Validation emails: 132
Vocabulary J: 2,048 | Train tokens: 47,540 | Validation tokens: 5,340
Context T: 128 | Token embedding d: 128


## 2. The decoder-only transformer

The model implements the operations we discussed. For one attention head, with input $Z$ of shape $(B,T,d)$:

$$
Q=ZW_Q,\quad K=ZW_K,\quad V=ZW_V
$$

Each of $Q$, $K$, $V$ is reshaped to $(B,H,T,d_h)$ where $d_h=d/H$. RoPE rotates the $Q$ and $K$ vectors based on their positions. **Causal attention** then computes

$$
A=\operatorname{softmax}\!\left(\frac{\widetilde Q\widetilde K^\top}{\sqrt{d_h}}+M\right),\qquad U=AV,
$$

where `M` masks future positions. PyTorch's `scaled_dot_product_attention` performs the score/softmax/weighted-average operations efficiently without explicitly returning `A`.

A block applies pre-normalization, residual connections, attention, and a GELU MLP:

$$
Z^{(0b)}=Z^{(0)}+\mathrm{Attention}(\mathrm{LayerNorm}(Z^{(0)})),\qquad
Z^{(1)}=Z^{(0b)}+\mathrm{MLP}(\mathrm{LayerNorm}(Z^{(0b)})).
$$

All blocks share their **architecture**, but each block learns **separate** weights. Within a block, the same weights are applied to all positions. This is a **dense decoder-only model** (not mixture-of-experts), with standard multi-head rather than grouped-query attention.

In [136]:
def apply_rope(x, cos, sin):
    """Rotate adjacent coordinate pairs; x shape (B,H,T,d_h)."""
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated = torch.stack((even * cos - odd * sin,
                           even * sin + odd * cos), dim=-1)
    return rotated.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        assert d % heads == 0
        self.heads = heads
        self.head_dim = d // heads
        assert self.head_dim % 2 == 0, "RoPE requires an even head dimension"
        self.W_Q = nn.Linear(d, d, bias=False)
        self.W_K = nn.Linear(d, d, bias=False)
        self.W_V = nn.Linear(d, d, bias=False)
        self.W_O = nn.Linear(d, d, bias=False)
        self.dropout_p = dropout
        inv_freq = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x):
        B, T, d = x.shape
        def split_heads(y):
            return y.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        q = split_heads(self.W_Q(x))  # (B,H,T,d_h)
        k = split_heads(self.W_K(x))  # (B,H,T,d_h)
        v = split_heads(self.W_V(x))  # (B,H,T,d_h)

        # Position-dependent rotations for Q and K, not V.
        positions = torch.arange(T, device=x.device, dtype=self.inv_freq.dtype)
        angles = torch.outer(positions, self.inv_freq)
        cos = angles.cos()[None, None, :, :]  # (1,1,T,d_h/2)
        sin = angles.sin()[None, None, :, :]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Causal mask ensures output at t cannot depend on tokens after t.
        u = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=(self.dropout_p if self.training else 0.0)
        )                                           # (B,H,T,d_h)
        u = u.transpose(1, 2).contiguous().view(B, T, d)
        return self.W_O(u)                          # (B,T,d)

class TransformerBlock(nn.Module):
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.GELU(),
            nn.Linear(4 * d, d),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size, d, heads, layers, context_length, dropout):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d)  # E: (J,d)
        self.blocks = nn.ModuleList([
            TransformerBlock(d, heads, dropout) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(d)
        self.output_projection = nn.Linear(d, vocab_size, bias=False)
        # Optional parameter sharing: output projection = input embedding matrix.
        self.output_projection.weight = self.token_embedding.weight
        # Small initial weights keep initial logits well scaled, including with tied embeddings.
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        if idx.shape[1] > self.context_length:
            raise ValueError("Sequence exceeds context length")
        x = self.token_embedding(idx)       # (B,T,d)
        for block in self.blocks:
            x = block(x)                   # (B,T,d)
        x = self.final_norm(x)
        logits = self.output_projection(x) # (B,T,J)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), targets.reshape(-1)
            )
        return logits, loss

model = TinyDecoderLM(
    vocab_size=J, d=EMBED_DIM, heads=NUM_HEADS, layers=NUM_LAYERS,
    context_length=block_size, dropout=DROPOUT
).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

Trainable parameters: 855,680


### A quick architectural check

The following test checks the shape and verifies causal masking: changing the **last** input token cannot alter logits computed at **earlier** positions. It is deliberately run in evaluation mode to disable dropout.

In [137]:
model.eval()
with torch.no_grad():
    probe1 = torch.randint(0, J, (2, 8), device=device)
    probe2 = probe1.clone()
    probe2[:, -1] = (probe2[:, -1] + 1) % J
    output1, _ = model(probe1)
    output2, _ = model(probe2)
    assert output1.shape == (2, 8, J)
    torch.testing.assert_close(output1[:, :-1, :], output2[:, :-1, :], atol=1e-5, rtol=1e-5)
print("Shape test and causal-masking test passed.")

Shape test and causal-masking test passed.


## 3. Train by next-token prediction

From a token stream `w_1,...,w_L`, each training example consists of a length-`T` input and the **same sequence shifted one token forward**. The cross-entropy loss uses all `T` positions in parallel. During inference, we use only the logits at the last position.

The train/validation split tests held-out emails, though shared authors, subject matter, and message templates mean that this is not a strong test of general-purpose language modeling. Training on CPU can take noticeably longer than on a GPU. Reduce `TRAIN_STEPS`, `BATCH_SIZE`, or `NUM_LAYERS` for a quicker run.

In [138]:
def get_batch(data, batch_size=BATCH_SIZE):
    # Offset one extra token for targets; choose contiguous spans.
    starts = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in starts.tolist()])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in starts.tolist()])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(eval_batches=8):
    model.eval()
    result = {}
    for name, stream in [("train", train_ids), ("validation", val_ids)]:
        values = []
        for _ in range(eval_batches):
            xb, yb = get_batch(stream)
            _, loss = model(xb, yb)
            values.append(loss.item())
        result[name] = sum(values) / len(values)
    model.train()
    return result

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
model.train()
for step in range(1, TRAIN_STEPS + 1):
    xb, yb = get_batch(train_ids)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    if step == 1 or step % EVAL_INTERVAL == 0 or step == TRAIN_STEPS:
        scores = estimate_loss()
        print(f"step {step:4d} | train loss {scores['train']:.3f} | validation loss {scores['validation']:.3f}")

step    1 | train loss 7.574 | validation loss 7.575
step   50 | train loss 6.372 | validation loss 6.412
step  100 | train loss 5.888 | validation loss 5.930
step  150 | train loss 5.481 | validation loss 5.616
step  200 | train loss 5.226 | validation loss 5.412
step  250 | train loss 5.080 | validation loss 5.278


## 4. Generate a continuation from a user-supplied prompt

`model.eval()` turns off dropout; the model produces logits for every position, but we sample from **only the last row**. We append the sampled token and repeat. `temperature` and `top_k` control the sampling distribution. The generation loop recomputes the context each time for clarity; production systems typically cache keys and values for speed.

The generated output may contain memorized fragments of emails. Treat it as sensitive until reviewed.

In [150]:
@torch.no_grad()
def generate_text(prompt, max_new_tokens=100, temperature=0.9, top_k=40):
    if not prompt:
        raise ValueError("Please supply a nonempty prompt")
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    model.eval()
    ids = tokenizer.encode(prompt).ids
    if not ids:
        raise ValueError("Prompt did not produce any tokens")
    generated = ids.copy()
    for _ in range(max_new_tokens):
        context = torch.tensor([generated[-block_size:]], device=device)
        logits, _ = model(context)
        last_logits = logits[:, -1, :] / temperature
        if top_k is not None:
            k = min(top_k, last_logits.shape[-1])
            threshold = torch.topk(last_logits, k, dim=-1).values[:, -1, None]
            last_logits = last_logits.masked_fill(last_logits < threshold, float("-inf"))
        probabilities = F.softmax(last_logits, dim=-1)
        next_id = int(torch.multinomial(probabilities, num_samples=1).item())
        if next_id == EOS_ID:
            break
        generated.append(next_id)
    return tokenizer.decode(generated, skip_special_tokens=True)

# Change this text, or uncomment the input() line to type a new prompt each run.
USER_PROMPT = "What was that noise?"
# USER_PROMPT = input("Enter a text prompt: ")
import textwrap
print(textwrap.fill(
    generate_text(
        USER_PROMPT, max_new_tokens=120, temperature=0.9, top_k=40
        ),width=50)
    )

What was that noise? The sun and one hand, and he,
a pp of the chapel, though my woods, but a man’s
eyes, as the forest. I was have not a stone. WeThe
donkeys was a de, with him that he said of the
courtyard, to’s stabed at it in my father’s reenty
old in me the creature. He had be, the old, the
small it.”


In [140]:
documents

,0
0,Will Stanger
1,318 N 300 E Provo UT 84606
2,425-830-7162
3,johnwbstanger@gmail.com
4,Approx. 32400 words
...,...
1319,I lay still and watched the ceiling.
1320,"The dream was still clear, which was unusual. ..."
1321,I was still thinking about it when young Fenwi...
1322,"""Please,"" I said."
